In [2]:

import csv
import random
import math

In [4]:
def loadcsv(filename):
    lines = csv.reader(open(filename, "r"))
    next(lines)
    dataset = list(lines)

    for i in range(len(dataset)):
        dataset[i] = [float(x) for x in dataset[i]]

    return dataset

In [5]:
def splitdataset(dataset, splitratio):
    trainsize = int(len(dataset) * splitratio)
    trainset = []
    copy = list(dataset)

    while len(trainset) < trainsize:
        index = random.randrange(len(copy))
        trainset.append(copy.pop(index))

    return [trainset, copy]

In [6]:
def separatebyclass(dataset):
    separated = {}

    for i in range(len(dataset)):
        vector = dataset[i]
        classvalue = vector[-1]

        if classvalue not in separated:
            separated[classvalue] = []

        separated[classvalue].append(vector)

    return separated

In [7]:
def mean(numbers):
    return sum(numbers) / float(len(numbers))

In [8]:
def stdev(numbers):
    avg = mean(numbers)

    variance = sum([(x - avg) ** 2 for x in numbers]) / float(len(numbers) - 1)

    return math.sqrt(variance)

In [9]:
def summarize(dataset):
    summaries = [(mean(attribute), stdev(attribute))
                 for attribute in zip(*dataset)]

    del summaries[-1]

    return summaries

In [10]:
def summarizebyclass(dataset):
    separated = separatebyclass(dataset)
    summaries = {}

    for classvalue, instances in separated.items():
        summaries[classvalue] = summarize(instances)

    return summaries

In [11]:
def calculateprobability(x, mean, stdev):
    exponent = math.exp(-((x - mean) ** 2 / (2 * stdev ** 2)))

    return (1 / (math.sqrt(2 * math.pi) * stdev)) * exponent

In [12]:
def calculateclassprobabilities(summaries, inputvector):
    probabilities = {}

    for classvalue, classsummaries in summaries.items():
        probabilities[classvalue] = 1

        for i in range(len(classsummaries)):
            meanvalue, stdevvalue = classsummaries[i]

            probabilities[classvalue] *= calculateprobability(
                inputvector[i], meanvalue, stdevvalue
            )

    return probabilities

In [13]:
def predict(summaries, inputvector):
    probabilities = calculateclassprobabilities(summaries, inputvector)

    bestlabel = None
    bestprob = -1

    for classvalue, probability in probabilities.items():
        if bestlabel is None or probability > bestprob:
            bestprob = probability
            bestlabel = classvalue

    return bestlabel

In [14]:
def getpredictions(summaries, testset):
    predictions = []

    for i in range(len(testset)):
        result = predict(summaries, testset[i])
        predictions.append(result)

    return predictions

In [15]:
def getaccuracy(testset, predictions):
    correct = 0

    for i in range(len(testset)):
        if testset[i][-1] == predictions[i]:
            correct += 1

    return (correct / float(len(testset))) * 100.0

In [16]:
filename = 'Naive-Bayes-Classification-Data.csv'

splitratio = 0.67

dataset = loadcsv(filename)

trainingset, testset = splitdataset(dataset, splitratio)

print("Split %d rows into train=%d and test=%d rows" %
      (len(dataset), len(trainingset), len(testset)))

summaries = summarizebyclass(trainingset)

predictions = getpredictions(summaries, testset)

accuracy = getaccuracy(testset, predictions)

print("Accuracy: %s%%" % accuracy)

Split 995 rows into train=666 and test=329 rows
Accuracy: 99.3920972644377%
